In [ ]:
from my_xso_model import model, model_setup_stability

param_dict={
    'Nutrient__value_init':0.07779513,
    'Phytoplankton__value_init':0.922204874680352,
}

with model:
    model_output = model_setup_stability.xsimlab.update_vars(
            input_vars=param_dict
        ).xsimlab.run()

In [ ]:
import numpy as np

from xso.parscans import run_xso_parscan

result_dataset = run_xso_parscan(
    model_file_name='my_xso_model',
    model_setup_name='model_setup_deriv',
    param_name='Nutrient__value_init',
    param_values=np.linspace(0.0001, 1.5, 20) ,
    param_name2='Phytoplankton__value_init',
    param_values2=np.linspace(0.0001, 1.5, 20) ,
    processes=20
)

# Display the result
if result_dataset is not None:
    print("\nFinal Scan Output Dataset:")
    print(result_dataset)

In [ ]:
import matplotlib.pyplot as plt

# --- 1. Get Steady State (SS) Values ---
# (Assuming 'model_output' is the result from your FSolver run)
N_ss = model_output.Nutrient__value.isel(time=1).values
P_ss = model_output.Phytoplankton__value.isel(time=1).values

# --- 2. Extract Vector Field Data ---
# (Assuming 'result_dataset' is from your DerivativeCalculator scan)
N_axis = result_dataset['Nutrient__value_init']
P_axis = result_dataset['Phytoplankton__value_init']
dN_dt = result_dataset['Nutrient__value'].isel(time=1)
dP_dt = result_dataset['Phytoplankton__value'].isel(time=1)


# --- 3. Create the Plot ---

print("Plotting vector field and steady state...")
fig, ax = plt.subplots(1, 1, figsize=(10, 8))

# Plot the streamplot (the vector field)
ax.streamplot(
    N_axis.values,    # x-coords
    P_axis.values,    # y-coords
    dN_dt.values,     # u (x-direction velocity)
    dP_dt.values,     # v (y-direction velocity)
    color='gray',
    density=1.5
)

# --- 4. ADD THE STEADY STATE ---
# Plot the coexistence steady state as a large, visible marker
ax.plot(
    N_ss, 
    P_ss, 
    'ro',  # 'r' = red, 'o' = circle marker
    markersize=10, 
    markeredgecolor='black', # Add a black edge for visibility
    label=f'Coexistence SS ({N_ss:.2f}, {P_ss:.2f})'
)


# --- 5. Format the Plot ---
ax.set_xlabel('Nutrient (N)')
ax.set_ylabel('Phytoplankton (P)')
ax.set_title('NP Chemostat Phase Portrait')
ax.set_xlim(N_axis.min(), N_axis.max())
ax.set_ylim(P_axis.min(), P_axis.max())
ax.legend() # Add a legend to label the points

plt.show()

In [ ]:
import numpy as np
from xso.parscans import run_xso_parscan
# Assuming your custom solvers are imported
# from custom_solvers import DerivativeCalculator, FSolver

# --- Scan 1: Get the Vector Field ---
# (Using the DerivativeCalculator you created)
print("Running vector field scan...")
vector_field_ds = run_xso_parscan(
    model_file_name='my_xso_model_deriv',
    param_name='Nutrient__value_init',
    param_values=np.linspace(0.0001, 1.5, 200), # Use the wider range
    param_name2='Phytoplankton__value_init',
    param_values2=np.linspace(0.0001, 1.5, 200), # Use the wider range
    processes=20
)

# --- Scan 2: Get the Converged Steady States ---
# (Using the default FSolver)
print("Running steady state scan...")
steady_state_ds = run_xso_parscan(
    model_file_name='my_xso_model_steadystate',
    param_name='Nutrient__value_init',
    param_values=np.linspace(0.0001, 1.5, 20),
    param_name2='Phytoplankton__value_init',
    param_values2=np.linspace(0.0001, 1.5, 20),
    processes=20
)

print("Scans complete. Plotting...")

In [ ]:
import matplotlib.pyplot as plt

# --- 1. Extract Vector Field Data ---
# (Using 'vector_field_ds' from Scan 1)
N_axis = vector_field_ds['Nutrient__value_init']
P_axis = vector_field_ds['Phytoplankton__value_init']
dN_dt = vector_field_ds['Nutrient__value'].isel(time=1)
dP_dt = vector_field_ds['Phytoplankton__value'].isel(time=1)

# --- 2. Extract ALL Steady State (SS) Data ---
# (Using 'steady_state_ds' from Scan 2)
# These are 2D arrays (shape 20x20)
N_ss_all = steady_state_ds['Nutrient__value'].isel(time=1)
P_ss_all = steady_state_ds['Phytoplankton__value'].isel(time=1)

# Flatten them into 1D arrays (of 400 points)
N_ss_flat = N_ss_all.values.flatten()
P_ss_flat = P_ss_all.values.flatten()

# --- 3. Create the Plot ---
print("Plotting...")
fig, ax = plt.subplots(1, 1, figsize=(10, 8))

# Plot the streamplot (the vector field)
ax.streamplot(
    N_axis.values,    # x-coords
    P_axis.values,    # y-coords
    dN_dt.values,     # u (x-direction velocity)
    dP_dt.values,     # v (y-direction velocity)
    color='gray',
    density=1.5
)

# --- 4. Plot ALL the Converged Steady States ---
# Use scatter to show all 400 converged points
ax.scatter(
    N_ss_flat, 
    P_ss_flat, 
    color='red',
    marker='.',  # Use small dots
    s=100,        # Small size
    alpha=0.5,   # Make them semi-transparent
    label='Converged Steady States (all 400 points)'
)

# --- 5. Format the Plot ---
ax.set_xlabel('Nutrient (N)')
ax.set_ylabel('Phytoplankton (P)')
ax.set_title('NP Chemostat Phase Portrait and Basins of Attraction')
ax.set_xlim(N_axis.min(), N_axis.max())
ax.set_ylim(P_axis.min(), P_axis.max())
ax.legend()

plt.show()